# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 1.028754085615041),
 (1, 1.028754085615041),
 (2, 1.028754085615041),
 (3, 1.028754085615041),
 (4, 1.028754085615041)]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('it', 18)]),
 (1, [('is', 18), ('what', 10)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.018980547052389163),
   (None, 0.022165485017636932),
   (None, 0.061561131561569615),
   (None, 0.21241307934096332),
   (None, 0.31926842384322096),
   (None, 0.32983580154392667),
   (None, 0.35272318894609234),
   (None, 0.3729364624474971),
   (None, 0.38048452379864106),
   (None, 0.39380503291082003),
   (None, 0.39575283423560903),
   (None, 0.439213763108795),
   (None, 0.47833359940770726),
   (None, 0.48404791015593696),
   (None, 0.49969831762641737)]),
 (1,
  [(None, 0.6207137477548006),
   (None, 0.6446942716658729),
   (None, 0.651222965231288),
   (None, 0.6772657231561845),
   (None, 0.7528647305867635),
   (None, 0.785704748911121),
   (None, 0.8052460617274902),
   (None, 0.8425020854383031),
   (None, 0.8851651863141086),
   (None, 0.9157882756728528),
   (None, 0.9332216362947846),
   (None, 0.9535479668720422),
   (None, 0.9677463048462734),
   (None, 0.9850345975051914),
   (None, 0.9917457900817522)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [3]:
import random
from functools import reduce

# Генератор данных: создает список случайных целых чисел
def data_generator(sample_size):
    return [random.randint(0, 100) for _ in range(sample_size)]

# Трансформация (Map): извлекает наибольший элемент из переданного списка
def extract_max(element_list):
    return max(element_list)

# Агрегация (Reduce): выбирает больший из двух результатов
def pick_larger(prev_max, current_max):
    return max(prev_max, current_max)

# Генерируем исходные данные
raw_data = data_generator(15)
print(f"Исходный набор: {raw_data}")

# Применяем "Map" операцию
mapped_results = map(extract_max, [raw_data])  # Оборачиваем в список для map

# Применяем "Reduce" операцию
final_max = reduce(pick_larger, mapped_results)

print(f"Максимальное значение: {final_max}")

Исходный набор: [37, 56, 28, 71, 61, 18, 44, 1, 59, 3, 1, 72, 89, 79, 97]
Максимальное значение: 97


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [6]:
import random
from typing import Iterator, Tuple

# Функция подготовки данных
def generate_samples(quantity):
    return [random.randint(0, 100) for _ in range(quantity)]

# Функция преобразования: добавляет метку-ключ к каждому числу
def add_group_label(value):
    return (0, value)  # Ключ 0 для всех элементов

# Функция агрегации: вычисляет среднее по группе
def calculate_average(key, values: Iterator[Tuple[int, int]]):
    total = 0
    elements_count = 0
    for _, val in values:
        total += val
        elements_count += 1
    yield total / elements_count

# Генерируем данные
data_series = generate_samples(100)
print("Сгенерированные значения:", data_series[:])

# Этап преобразования
mapped_data = list(map(add_group_label, data_series))
print("После Map:", mapped_data[:])

# Этап агрегации (группируем по ключу 0)
final_result = list(calculate_average(0, mapped_data))
print(f"Среднее арифметическое: {final_result[0]:.2f}")

Сгенерированные значения: [15, 5, 40, 0, 63, 20, 49, 29, 72, 9, 26, 73, 89, 64, 94, 84, 16, 89, 46, 55, 92, 100, 12, 69, 77, 37, 92, 65, 94, 0, 63, 90, 73, 7, 6, 73, 60, 17, 12, 7, 74, 2, 2, 26, 72, 97, 30, 38, 24, 90, 48, 68, 9, 49, 50, 42, 40, 46, 2, 8, 41, 31, 16, 5, 38, 78, 15, 3, 96, 46, 91, 92, 74, 87, 65, 45, 48, 41, 99, 35, 39, 88, 1, 2, 49, 56, 32, 21, 13, 95, 95, 51, 7, 50, 30, 5, 90, 78, 27, 65]
После Map: [(0, 15), (0, 5), (0, 40), (0, 0), (0, 63), (0, 20), (0, 49), (0, 29), (0, 72), (0, 9), (0, 26), (0, 73), (0, 89), (0, 64), (0, 94), (0, 84), (0, 16), (0, 89), (0, 46), (0, 55), (0, 92), (0, 100), (0, 12), (0, 69), (0, 77), (0, 37), (0, 92), (0, 65), (0, 94), (0, 0), (0, 63), (0, 90), (0, 73), (0, 7), (0, 6), (0, 73), (0, 60), (0, 17), (0, 12), (0, 7), (0, 74), (0, 2), (0, 2), (0, 26), (0, 72), (0, 97), (0, 30), (0, 38), (0, 24), (0, 90), (0, 48), (0, 68), (0, 9), (0, 49), (0, 50), (0, 42), (0, 40), (0, 46), (0, 2), (0, 8), (0, 41), (0, 31), (0, 16), (0, 5), (0, 38), (0, 7

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [12]:
import random
from typing import Iterator, List, Tuple

def flatten(nested_iterable):
    """Преобразует вложенный итерируемый объект в плоский"""
    for iterable in nested_iterable:
        for element in iterable:
            yield element

# Функция Map: создает пару (ключ=1, значение=число)
def assign_group(number):
    return (1, number)

# Функция Reduce: вычисляет среднее для группы
def compute_average(key, values: Iterator[int]):
    total = 0
    count = 0
    for val in values:
        total += val
        count += 1
    yield total / count if count > 0 else 0

# Генератор данных
def generate_numbers(amount):
    return [random.randint(0, 100) for _ in range(amount)]

# Группировка по ключу с сортировкой
def sort_and_group(pairs: List[Tuple[int, int]]):
    groups = {}
    for k, v in sorted(pairs, key=lambda x: x[0]):
        groups.setdefault(k, []).append(v)
    return list(groups.items())

# Выполнение
raw_data = generate_numbers(100)
print(f"Сгенерировано {len(raw_data)} чисел")

mapped = list(map(assign_group, raw_data))
print(f"После Map: {len(mapped)} пар (ключ, значение)")

shuffled = sort_and_group(mapped)
print(f"После группировки:", shuffled)

reduced = list(flatten(map(lambda x: compute_average(*x), shuffled)))
print(f"Результат (среднее): {reduced[0]:.2f}")

Сгенерировано 100 чисел
После Map: 100 пар (ключ, значение)
После группировки: [(1, [22, 1, 6, 86, 46, 70, 61, 49, 46, 76, 93, 45, 97, 49, 25, 11, 51, 29, 17, 32, 59, 6, 1, 23, 76, 71, 65, 59, 67, 99, 96, 49, 94, 14, 79, 46, 49, 81, 31, 70, 91, 37, 73, 93, 35, 25, 12, 100, 98, 30, 60, 33, 69, 68, 19, 32, 28, 74, 87, 15, 68, 51, 89, 96, 4, 80, 95, 21, 21, 90, 39, 77, 90, 27, 46, 68, 48, 0, 86, 82, 22, 65, 7, 81, 67, 79, 92, 50, 23, 43, 72, 24, 20, 83, 81, 91, 71, 82, 36, 89])]
Результат (среднее): 54.82


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [19]:
def flatten(nested_collection):
    # Преобразование вложенных структур в плоский поток данных
    for sub_collection in nested_collection:
        for element in sub_collection:
            yield element

def groupbykey(data_pairs):
    # Объединение значений по одинаковым ключам
    grouped_result = {}
    for key, value in data_pairs:
        grouped_result.setdefault(key, []).append(value)
    return list(grouped_result.items())

def groupbykey_distributed(map_partitions, PARTITIONER):
    # Распределенная версия группировки с учетом партиций
    global reducers
    partitions_container = [dict() for _ in range(reducers)]
    for single_partition in map_partitions:
        for current_key, current_value in single_partition:
            target_partition = partitions_container[PARTITIONER(current_key)]
            target_partition.setdefault(current_key, []).append(current_value)
    return [(partition_index, sorted(partition.items(), key=lambda x: x[0]))
            for (partition_index, partition) in enumerate(partitions_container)]

def PARTITIONER(key_object):
    # Определение номера партиции на основе хеша ключа
    global reducers
    return hash(key_object) % reducers

# Основная функция распределенной обработки
def MapReduceDistributed(INPUT_FORMAT, MAP_FUNC, REDUCE_FUNC, PARTITIONER=PARTITIONER, COMBINER_FUNC=None):

    # Этап MAP: преобразование входных данных в промежуточные пары
    map_stage_results = map(
        lambda record_reader: flatten(map(lambda record: MAP_FUNC(*record), record_reader)),
        INPUT_FORMAT()
    )

    # Этап COMBINE (опционально): локальная агрегация для уменьшения данных
    if COMBINER_FUNC is not None:
        map_stage_results = map(
            lambda map_output: flatten(map(
                lambda grouped_pair: COMBINER_FUNC(*grouped_pair),
                groupbykey(map_output)
            )),
            map_stage_results
        )

    # Этап SHUFFLE: распределение данных по редукторам
    reduce_stage_input = groupbykey_distributed(map_stage_results, PARTITIONER)

    # Подсчет переданных по сети данных
    total_network_pairs = sum([
        len(values_list) for (_, values_list) in flatten([
            partition_data for (_, partition_data) in reduce_stage_input
        ])
    ])
    print(f"{total_network_pairs} пар ключ-значение передано по сети.")

    # Этап REDUCE: финальная обработка каждой группы
    reduce_stage_output = map(
        lambda reducer_data: (
            reducer_data[0],
            flatten(map(lambda input_group: REDUCE_FUNC(*input_group), reducer_data[1]))
        ),
        reduce_stage_input
    )

    return reduce_stage_output


from typing import Iterator
import numpy as np

# Исходные текстовые документы
doc1 = """
hello world hello
hello world hello
hello world hello"""
doc2 = """
world hello
world hello"""
doc3 = """
hello python world"""
document_collection = [doc1, doc2, doc3, doc1, doc2, doc3]

map_tasks = 3
reduce_tasks = 2

def INPUT_FORMAT():
    # Формирование входных сплитов для мапперов
    global map_tasks

    # Читатель записей для одного сплита
    def RECORD_READER(data_split):
        for (doc_number, document_text) in enumerate(data_split):
            for (line_number, line_content) in enumerate(document_text.split('\n')):
                if line_content.strip():  # пропускаем пустые строки
                    yield (f"{doc_number}:{line_number}", line_content)

    # Размер одного сплита
    split_size = int(np.ceil(len(document_collection) / map_tasks))

    # Генерация сплитов
    for start_idx in range(0, len(document_collection), split_size):
        yield RECORD_READER(document_collection[start_idx:start_idx + split_size])

def MAP_FUNC(document_id: str, text_line: str):
    # Извлечение слов из строки документа
    for word in text_line.split(" "):
        if word:  # игнорируем пустые строки
            yield (word, word)

def REDUCE_FUNC(unique_word: str, occurrences: Iterator[str]):
    # Возвращаем только уникальное слово (дубликаты отбрасываются)
    yield unique_word

# Запуск MapReduce задачи
final_result = MapReduceDistributed(INPUT_FORMAT, MAP_FUNC, REDUCE_FUNC, COMBINER_FUNC=None)
formatted_result = [(partition_id, list(partition_data)) for (partition_id, partition_data) in final_result]
formatted_result

32 пар ключ-значение передано по сети.


[(0, ['hello', 'world']), (1, ['python'])]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [23]:
from collections import defaultdict
import random

# Функция для реализации MAP операции
def MAP(el_list):
    mapped_result = defaultdict(list)
    for t in el_list:
        if C(t):  # Предикат C - выбираем только кортежи, удовлетворяющие условию
            mapped_result[t].append(t)
    return mapped_result.items()

# Функция для реализации REDUCE операции
def REDUCE(mapped_items):
    reduced_result = []
    print("Промежуточный результат (данные после MAP, переданные в REDUCE)")
    print(mapped_items)

    for values in mapped_items:
        for value in values:
            reduced_result.append(value)  # Функция идентичности: возвращаем то же значение, что получили на вход
    return reduced_result

# Предикат для фильтрации кортежей с четным первым элементом
def C(t):
    return t[0] % 2 == 0

# Генерация случайных записей (кортежей формата (x, y, z))
def RECORDREADER(count):
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(count)]

# Создание набора записей
print("Исходные данные (сгенерированные кортежи)")
record = RECORDREADER(5)
print(record)
print()

# Разбиение записей на равные части
print("Разбиение данных на части")
part_count = 5
record_partitioned = [record[d:d + part_count] for d in range(0, len(record), part_count)]
print(f"Количество частей: {len(record_partitioned)}")
for i, part in enumerate(record_partitioned):
    print(f"Часть {i+1}: {part}")
print()

# Применение операции MAP и последующей REDUCE к разделенным записям
print("Результат выполнения MapReduce")
final_result = REDUCE(list(map(lambda x: MAP(x), record_partitioned)))
print("\nФинальный результат (кортежи, прошедшие фильтрацию)")
print(final_result)

Исходные данные (сгенерированные кортежи)
[(14, 80, 27), (75, 87, 73), (50, 45, 69), (69, 36, 73), (51, 24, 46)]

Разбиение данных на части
Количество частей: 1
Часть 1: [(14, 80, 27), (75, 87, 73), (50, 45, 69), (69, 36, 73), (51, 24, 46)]

Результат выполнения MapReduce
Промежуточный результат (данные после MAP, переданные в REDUCE)
[dict_items([((14, 80, 27), [(14, 80, 27)]), ((50, 45, 69), [(50, 45, 69)])])]

Финальный результат (кортежи, прошедшие фильтрацию)
[((14, 80, 27), [(14, 80, 27)]), ((50, 45, 69), [(50, 45, 69)])]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [27]:
from collections import defaultdict
import random

# Функция для реализации MAP операции
def MAP(el_list):
    mapped_result = defaultdict(list)
    for t in el_list:
        if C(t):  # Предикат C - выбираем только кортежи, удовлетворяющие условию
            mapped_result[t].append(t)
    return mapped_result.items()

# Функция для реализации REDUCE операции
def REDUCE(mapped_items):
    reduced_result = []
    print("Промежуточный результат (данные после MAP, переданные в REDUCE):")
    for i, items in enumerate(mapped_items, 1):
        print(f"  Часть {i}: {dict(items)}")

    for values in mapped_items:
        for value in values:
            reduced_result.append(value)  # Функция идентичности: возвращаем то же значение, что получили на вход
    return reduced_result

# Предикат для фильтрации кортежей с четным первым элементом
def C(t):
    return t[0] % 2 == 0

# Генерация случайных записей (кортежей формата (x, y, z))
def RECORDREADER(count):
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(count)]

record = RECORDREADER(5)
print("Сгенерированные кортежи:")
for i, t in enumerate(record, 1):
    print(f"  {i}. {t}")
print()

part_count = 5
record_partitioned = [record[d:d + part_count] for d in range(0, len(record), part_count)]
print(f"Количество частей: {len(record_partitioned)}")
for i, part in enumerate(record_partitioned, 1):
    print(f"\nЧасть {i}:")
    for j, t in enumerate(part, 1):
        print(f"  {j}. {t}")
print()

print("Применяем предикат C (первый элемент четный) к каждому кортежу")
map_results = list(map(lambda x: MAP(x), record_partitioned))
print()


final_result = REDUCE(map_results)
print()

print("Кортежи, прошедшие фильтрацию (первый элемент четный):")
if final_result:
    for i, t in enumerate(final_result, 1):
        print(f"  {i}. {t}")
else:
    print("Нет кортежей, удовлетворяющих условию")

print("\nСТАТИСТИКА:")
print(f"  Всего сгенерировано кортежей: {len(record)}")
print(f"  Кортежей, удовлетворяющих условию: {len(final_result)}")
if len(record) > 0:
    print(f"  Процент прошедших фильтр: {len(final_result)/len(record)*100:.1f}%")

Сгенерированные кортежи:
  1. (81, 47, 45)
  2. (68, 69, 45)
  3. (12, 73, 38)
  4. (38, 38, 53)
  5. (48, 95, 77)

Количество частей: 1

Часть 1:
  1. (81, 47, 45)
  2. (68, 69, 45)
  3. (12, 73, 38)
  4. (38, 38, 53)
  5. (48, 95, 77)

Применяем предикат C (первый элемент четный) к каждому кортежу

Промежуточный результат (данные после MAP, переданные в REDUCE):
  Часть 1: {(68, 69, 45): [(68, 69, 45)], (12, 73, 38): [(12, 73, 38)], (38, 38, 53): [(38, 38, 53)], (48, 95, 77): [(48, 95, 77)]}

Кортежи, прошедшие фильтрацию (первый элемент четный):
  1. ((68, 69, 45), [(68, 69, 45)])
  2. ((12, 73, 38), [(12, 73, 38)])
  3. ((38, 38, 53), [(38, 38, 53)])
  4. ((48, 95, 77), [(48, 95, 77)])

СТАТИСТИКА:
  Всего сгенерировано кортежей: 5
  Кортежей, удовлетворяющих условию: 4
  Процент прошедших фильтр: 80.0%


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [31]:
import random
from typing import Iterator, NamedTuple

def MAP(t):
    # Функция MAP возвращает кортеж из входного элемента и этого же элемента
    return (t, t)

def REDUCE(key, values: Iterator[NamedTuple]):
    # Функция REDUCE возвращает кортеж с ключом и значением равными ключу
    return (key, key)

def RECORDREADER(count):
    # Генерирует список кортежей случайных чисел
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for i in range(count)]

def group_by_key(iterable):
    # Группировка значений по ключу
    grouped_data = {}
    for (key, value) in iterable:
        grouped_data[key] = grouped_data.get(key, []) + [value]
    return grouped_data.items()

print("ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ")
sample_data = RECORDREADER(5)
print("Пример сгенерированных кортежей (первые 5):")
for i, item in enumerate(sample_data, 1):
    print(f"  {i}. {item}")
print()

print("ГЕНЕРАЦИЯ ПОЛНОГО НАБОРА ДАННЫХ")
full_data = RECORDREADER(100)
print(f"Сгенерировано {len(full_data)} кортежей")
print("Пример первых 5:")
for i, item in enumerate(full_data[:5], 1):
    print(f"  {i}. {item}")
print()

print("MAP-ФАЗА")
map_output = list(map(lambda x: MAP(x), full_data))
print(f"Результат MAP: {len(map_output)} пар (ключ, значение)")
print("Пример первых 5 результатов MAP:")
for i, pair in enumerate(map_output[:5], 1):
    print(f"  {i}. {pair}")
print()

print("SHUFFLE-ФАЗА (ГРУППИРОВКА ПО КЛЮЧУ)")
shuffle_output = list(group_by_key(map_output))
print(f"Результат SHUFFLE: {len(shuffle_output)} уникальных групп")
print("Статистика по группам (первые 5):")
for i, (key, values) in enumerate(shuffle_output[:5], 1):
    print(f"  Группа {i}: ключ {key} -> {len(values)} вхождений")
print()

print("REDUCE-ФАЗА")
reduce_output = list(map(lambda x: REDUCE(*x)[0], shuffle_output))
print(f"Результат REDUCE: {len(reduce_output)} уникальных элементов")
print("Первые 10 уникальных кортежей:")
for i, item in enumerate(reduce_output[:10], 1):
    print(f"  {i}. {item}")

ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ
Пример сгенерированных кортежей (первые 5):
  1. (61, 50, 4)
  2. (15, 61, 58)
  3. (79, 41, 91)
  4. (18, 7, 48)
  5. (61, 16, 54)

ГЕНЕРАЦИЯ ПОЛНОГО НАБОРА ДАННЫХ
Сгенерировано 100 кортежей
Пример первых 5:
  1. (48, 97, 12)
  2. (85, 43, 90)
  3. (10, 68, 68)
  4. (41, 49, 51)
  5. (70, 35, 14)

MAP-ФАЗА
Результат MAP: 100 пар (ключ, значение)
Пример первых 5 результатов MAP:
  1. ((48, 97, 12), (48, 97, 12))
  2. ((85, 43, 90), (85, 43, 90))
  3. ((10, 68, 68), (10, 68, 68))
  4. ((41, 49, 51), (41, 49, 51))
  5. ((70, 35, 14), (70, 35, 14))

SHUFFLE-ФАЗА (ГРУППИРОВКА ПО КЛЮЧУ)
Результат SHUFFLE: 100 уникальных групп
Статистика по группам (первые 5):
  Группа 1: ключ (48, 97, 12) -> 1 вхождений
  Группа 2: ключ (85, 43, 90) -> 1 вхождений
  Группа 3: ключ (10, 68, 68) -> 1 вхождений
  Группа 4: ключ (41, 49, 51) -> 1 вхождений
  Группа 5: ключ (70, 35, 14) -> 1 вхождений

REDUCE-ФАЗА
Результат REDUCE: 100 уникальных элементов
Первые 10 уникальных кортежей:

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [34]:
import random
from typing import Iterator, NamedTuple

def MAP(t):
    # Функция MAP возвращает кортеж из входного элемента и этого же элемента
    return (t, t)

def REDUCE(key, values: Iterator[NamedTuple]):
    # Если количество значений равно 2, значит элемент есть в обоих множествах
    if len(values) == 2:
        return (key, key)  # Возвращаем элемент, который есть в пересечении

def RECORDREADER(count):
    # Генерирует список кортежей случайных чисел (значения от 0 до 3)
    return [(random.randint(0, 3), random.randint(0, 3)) for i in range(count)]

def group_by_key(iterable):
    # Группировка значений по ключу
    grouped_data = {}
    for (key, value) in iterable:
        grouped_data[key] = grouped_data.get(key, []) + [value]
    return grouped_data.items()

print("ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ")
sample_data = RECORDREADER(10)
print("Пример сгенерированных кортежей (первые 10):")
for i, item in enumerate(sample_data, 1):
    print(f"  {i}. {item}")
print()

print("ГЕНЕРАЦИЯ ПОЛНОГО НАБОРА ДАННЫХ (ДВА МНОЖЕСТВА)")
# Первое множество
set1 = RECORDREADER(50)
# Второе множество
set2 = RECORDREADER(50)
# Объединяем для обработки
full_data = set1 + set2
print(f"Всего сгенерировано кортежей: {len(full_data)} (по 50 из каждого множества)")
print("Пример первых 10 из общего списка:")
for i, item in enumerate(full_data[:10], 1):
    print(f"  {i}. {item}")
print()

print("MAP-ФАЗА")
map_output = list(map(lambda x: MAP(x), full_data))
print(f"Результат MAP: {len(map_output)} пар (ключ, значение)")
print("Пример первых 10 результатов MAP:")
for i, pair in enumerate(map_output[:10], 1):
    print(f"  {i}. {pair}")
print()

print("SHUFFLE-ФАЗА (ГРУППИРОВКА ПО КЛЮЧУ)")
shuffle_output = list(group_by_key(map_output))
print(f"Результат SHUFFLE: {len(shuffle_output)} уникальных групп")
print("Информация по группам (первые 10):")
for i, (key, values) in enumerate(shuffle_output[:10], 1):
    print(f"  Группа {i}: ключ {key} -> {len(values)} вхождений")
print()

print("REDUCE-ФАЗА (ПОИСК ПЕРЕСЕЧЕНИЯ)")
print("(Элемент попадает в результат, если встречается ровно 2 раза - по одному из каждого множества)")
reduce_output = [el[0] for el in list(map(lambda x: REDUCE(*x), shuffle_output)) if el is not None]
print(f"Результат REDUCE: {len(reduce_output)} элементов в пересечении")
print("Элементы, присутствующие в обоих множествах:")
if reduce_output:
    for i, item in enumerate(reduce_output, 1):
        print(f"  {i}. {item}")
else:
    print("  Пересечение пусто")
print()

print("ИТОГОВАЯ СТАТИСТИКА")
print(f"  Размер первого множества: {len(set1)}")
print(f"  Размер второго множества: {len(set2)}")
print(f"  Размер пересечения: {len(reduce_output)}")
if len(set1) > 0 and len(set2) > 0:
    print(f"  Относительный размер пересечения: {len(reduce_output)/min(len(set1), len(set2))*100:.1f}% от меньшего множества")

ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ
Пример сгенерированных кортежей (первые 10):
  1. (0, 3)
  2. (0, 1)
  3. (1, 3)
  4. (1, 3)
  5. (3, 0)
  6. (2, 1)
  7. (2, 3)
  8. (3, 3)
  9. (3, 2)
  10. (2, 2)

ГЕНЕРАЦИЯ ПОЛНОГО НАБОРА ДАННЫХ (ДВА МНОЖЕСТВА)
Всего сгенерировано кортежей: 100 (по 50 из каждого множества)
Пример первых 10 из общего списка:
  1. (1, 2)
  2. (1, 2)
  3. (1, 1)
  4. (2, 0)
  5. (1, 1)
  6. (1, 1)
  7. (0, 0)
  8. (3, 2)
  9. (2, 2)
  10. (2, 1)

MAP-ФАЗА
Результат MAP: 100 пар (ключ, значение)
Пример первых 10 результатов MAP:
  1. ((1, 2), (1, 2))
  2. ((1, 2), (1, 2))
  3. ((1, 1), (1, 1))
  4. ((2, 0), (2, 0))
  5. ((1, 1), (1, 1))
  6. ((1, 1), (1, 1))
  7. ((0, 0), (0, 0))
  8. ((3, 2), (3, 2))
  9. ((2, 2), (2, 2))
  10. ((2, 1), (2, 1))

SHUFFLE-ФАЗА (ГРУППИРОВКА ПО КЛЮЧУ)
Результат SHUFFLE: 16 уникальных групп
Информация по группам (первые 10):
  Группа 1: ключ (1, 2) -> 5 вхождений
  Группа 2: ключ (1, 1) -> 6 вхождений
  Группа 3: ключ (2, 0) -> 7 вхождений
  Групп

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [37]:
import random
from typing import Iterator, NamedTuple

# Идентификаторы двух отношений (R и S)
relation_ids = [1, 2]  # 1 - первое отношение (R), 2 - второе отношение (S)

class DataTuple:  # Класс для хранения данных и принадлежности к отношению
    def __init__(self, data: tuple, relation: int):
        self.data = data
        self.relation = relation  # Какому отношению принадлежит кортеж (1 или 2)

def generate_random_tuple(length):
    # Генерация случайного кортежа указанной длины
    data = tuple([(random.randint(0, 3), random.randint(0, 3)) for _ in range(length)])
    # Случайный выбор отношения (R или S)
    relation = relation_ids[random.randint(0, len(relation_ids) - 1)]
    return DataTuple(data, relation)

def RECORDREADER(count):
    # Генерация списка случайных кортежей
    return [generate_random_tuple(3) for _ in range(count)]

def MAP(t: DataTuple):
    # Функция MAP возвращает пару (данные, идентификатор отношения)
    return (t.data, t.relation)

def REDUCE(key, values: Iterator[NamedTuple]):
    # Если все значения для данного ключа равны 1 (только из R), то кортеж принадлежит R \ S
    if values == [relation_ids[0]]:  # Проверяем, есть ли только идентификатор R
        return (key, key)  # Возвращаем кортеж как результат разности

def group_by_key(iterable):
    # Группировка значений по ключу
    groups = {}
    for (key, value) in iterable:
        groups[key] = groups.get(key, []) + [value]
    return groups.items()

print("ГЕНЕРАЦИЯ ДАННЫХ ДЛЯ ДВУХ ОТНОШЕНИЙ (R и S)")
# Генерируем данные для обоих отношений
all_data = RECORDREADER(100)
print(f"Всего сгенерировано кортежей: {len(all_data)}")

# Подсчет статистики по отношениям
r_count = sum(1 for item in all_data if item.relation == 1)
s_count = sum(1 for item in all_data if item.relation == 2)
print(f"  Кортежей в отношении R (id=1): {r_count}")
print(f"  Кортежей в отношении S (id=2): {s_count}")
print()

print("ПРИМЕР СГЕНЕРИРОВАННЫХ ДАННЫХ (первые 10)")
for i, item in enumerate(all_data[:10], 1):
    print(f"  {i}. Данные: {item.data}, отношение: {item.relation} ({'R' if item.relation == 1 else 'S'})")
print()

print("MAP-ФАЗА")
map_output = list(map(lambda x: MAP(x), all_data))
print(f"Результат MAP: {len(map_output)} пар (ключ, значение)")
print("Пример первых 10 результатов MAP:")
for i, pair in enumerate(map_output[:10], 1):
    print(f"  {i}. {pair}")
print()

print("SHUFFLE-ФАЗА (ГРУППИРОВКА ПО ДАННЫМ)")
shuffle_output = list(group_by_key(map_output))
print(f"Результат SHUFFLE: {len(shuffle_output)} уникальных групп")
print("Информация по группам (первые 10):")
for i, (key, values) in enumerate(shuffle_output[:10], 1):
    rels = set(values)
    rel_names = [f"R({v})" if v == 1 else f"S({v})" for v in rels]
    print(f"  Группа {i}: данные {key} -> встречается в: {', '.join(rel_names)} ({len(values)} вхождений)")
print()

print("REDUCE-ФАЗА (РАЗНОСТЬ R \\ S)")
print("(Кортеж попадает в результат, если встречается ТОЛЬКО в отношении R)")
reduce_output = [el[0] for el in list(map(lambda x: REDUCE(*x), shuffle_output)) if el is not None]
print(f"Результат REDUCE: {len(reduce_output)} кортежей в разности R \\ S")
print("Кортежи, принадлежащие только отношению R:")
if reduce_output:
    for i, item in enumerate(reduce_output, 1):
        print(f"  {i}. {item}")
else:
    print("  Разность пуста (нет кортежей, принадлежащих только R)")
print()

print("ИТОГОВАЯ СТАТИСТИКА")
print(f"  Всего уникальных кортежей: {len(shuffle_output)}")
print(f"  Кортежи, встречающиеся только в R: {len(reduce_output)}")
print(f"  Кортежи, встречающиеся только в S: {sum(1 for k, v in shuffle_output if set(v) == {2})}")
print(f"  Кортежи, встречающиеся в обоих отношениях: {sum(1 for k, v in shuffle_output if set(v) == {1, 2})}")

ГЕНЕРАЦИЯ ДАННЫХ ДЛЯ ДВУХ ОТНОШЕНИЙ (R и S)
Всего сгенерировано кортежей: 100
  Кортежей в отношении R (id=1): 53
  Кортежей в отношении S (id=2): 47

ПРИМЕР СГЕНЕРИРОВАННЫХ ДАННЫХ (первые 10)
  1. Данные: ((1, 0), (2, 3), (1, 3)), отношение: 2 (S)
  2. Данные: ((2, 2), (2, 0), (3, 1)), отношение: 2 (S)
  3. Данные: ((3, 3), (3, 3), (3, 1)), отношение: 2 (S)
  4. Данные: ((1, 3), (2, 2), (2, 3)), отношение: 1 (R)
  5. Данные: ((3, 1), (2, 1), (0, 1)), отношение: 2 (S)
  6. Данные: ((1, 0), (2, 0), (1, 3)), отношение: 2 (S)
  7. Данные: ((2, 1), (0, 3), (0, 1)), отношение: 2 (S)
  8. Данные: ((1, 1), (1, 1), (3, 0)), отношение: 2 (S)
  9. Данные: ((2, 1), (1, 3), (1, 3)), отношение: 2 (S)
  10. Данные: ((3, 3), (3, 0), (0, 2)), отношение: 2 (S)

MAP-ФАЗА
Результат MAP: 100 пар (ключ, значение)
Пример первых 10 результатов MAP:
  1. (((1, 0), (2, 3), (1, 3)), 2)
  2. (((2, 2), (2, 0), (3, 1)), 2)
  3. (((3, 3), (3, 3), (3, 1)), 2)
  4. (((1, 3), (2, 2), (2, 3)), 1)
  5. (((3, 1), (2, 1),

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [44]:
import random
from typing import Iterator, NamedTuple

# Идентификаторы двух отношений для естественного соединения
relation_ids = [1, 2]  # 1 - отношение R(a,b), 2 - отношение S(b,c)

class TupleData:
    def __init__(self, data: tuple, relation: int):
        self.data = data
        self.relation = relation

def generate_random_tuple():
    data = (random.randint(0, 3), random.randint(0, 3))
    relation = relation_ids[random.randint(0, len(relation_ids) - 1)]
    return TupleData(data, relation)

def RECORDREADER(count):
    return [generate_random_tuple() for _ in range(count)]

def MAP(t: TupleData):
    if t.relation == relation_ids[0]:  # R(a,b)
        return (t.data[1], (t.relation, t.data[0]))  # ключ = b, значение = (R, a)
    else:  # S(b,c)
        return (t.data[0], (t.relation, t.data[1]))  # ключ = b, значение = (S, c)

def REDUCE(key, values: Iterator[NamedTuple]):
    result = []
    for val in values:
        result.append((val[0], key, val[1]))
    return result

def group_by_key(iterable):
    groups = {}
    for (key, value) in iterable:
        groups[key] = groups.get(key, []) + [value]
    return groups.items()

# Создание списка случайных кортежей
record = RECORDREADER(100)

# Применение функции MAP к каждому элементу
map_output = list(map(lambda x: MAP(x), RECORDREADER(100)))
print("РЕЗУЛЬТАТ MAP (пары ключ-значение после преобразования):")
print(map_output)
print()

# Группировка данных по ключу
shuffle_output = group_by_key(map_output)
shuffle_output = list(shuffle_output)
print("РЕЗУЛЬТАТ SHUFFLE (группировка по ключам):")
print(shuffle_output)
print()

# Применение функции REDUCE к каждой группе
reduce_output = list(map(lambda x: REDUCE(*x), shuffle_output))
print("РЕЗУЛЬТАТ REDUCE (соединенные кортежи):")
print(reduce_output)

РЕЗУЛЬТАТ MAP (пары ключ-значение после преобразования):
[(3, (1, 1)), (2, (2, 0)), (3, (1, 3)), (0, (2, 1)), (0, (1, 3)), (1, (1, 1)), (1, (1, 3)), (2, (2, 3)), (2, (1, 1)), (0, (2, 1)), (1, (2, 1)), (0, (2, 3)), (2, (1, 1)), (2, (1, 2)), (3, (1, 1)), (3, (1, 2)), (2, (1, 0)), (0, (2, 3)), (3, (1, 0)), (1, (1, 3)), (3, (2, 2)), (2, (2, 3)), (3, (2, 1)), (1, (2, 3)), (3, (1, 3)), (0, (2, 3)), (2, (2, 3)), (0, (2, 1)), (0, (2, 3)), (2, (1, 2)), (2, (2, 1)), (1, (1, 3)), (0, (2, 1)), (2, (1, 0)), (3, (2, 3)), (2, (1, 0)), (3, (2, 1)), (0, (1, 1)), (0, (2, 2)), (1, (1, 3)), (1, (2, 3)), (2, (2, 2)), (0, (2, 1)), (3, (1, 0)), (1, (2, 1)), (1, (2, 1)), (2, (2, 1)), (2, (1, 2)), (0, (1, 1)), (1, (1, 1)), (2, (1, 1)), (1, (1, 0)), (2, (1, 2)), (3, (1, 2)), (2, (1, 3)), (2, (1, 0)), (0, (2, 0)), (3, (2, 1)), (2, (2, 2)), (0, (1, 2)), (3, (2, 3)), (3, (2, 3)), (1, (1, 2)), (3, (1, 3)), (3, (1, 1)), (1, (2, 0)), (0, (1, 3)), (3, (1, 2)), (0, (2, 2)), (1, (2, 1)), (1, (1, 2)), (1, (1, 0)), (1, (2

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [49]:
import random
from typing import Iterator, NamedTuple

def get_random_tuple():
    # Функция для генерации случайного кортежа с тремя случайными значениями от 0 до 3
    return (random.randint(0, 3), random.randint(0, 3), random.randint(0, 3))

def RECORDREADER(count):
    # Функция для создания списка случайных кортежей заданного количества
    return [get_random_tuple() for i in range(count)]

def MAP(t: Tuple):
    # Функция для преобразования входного кортежа, возвращающая первые два элемента кортежа
    return (t[0], t[1])

def tetta(values):
    # Функция для вычисления суммы значений в переданном списке
    return sum(values)

def REDUCE(key, values: Iterator[NamedTuple]):
    # Функция для вычисления суммы значений и возврата пары (ключ, сумма)
    x = tetta(values)
    return (key, x)

def group_by_key(iterable):
    # Функция для группировки данных по ключу
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

# Создание списка случайных кортежей
record = RECORDREADER(100)
print("ИСХОДНЫЕ ДАННЫЕ:", record[:])
print()

# Применение функции MAP к каждому элементу
map_output = list(map(lambda x: MAP(x), RECORDREADER(100)))
print("РЕЗУЛЬТАТ MAP (пары ключ-значение):")
print("Ключ (первый элемент), Значение (второй элемент)")
print(map_output[:])
print()

# Группировка данных по ключу
shuffle_output = group_by_key(map_output)
shuffle_output = list(shuffle_output)
print("РЕЗУЛЬТАТ SHUFFLE (группировка по ключам):")
print("Для каждого ключа показан список значений")
print(shuffle_output[:])
print()

# Применение функции REDUCE к каждой группе
reduce_output = list(map(lambda x: REDUCE(*x), shuffle_output))
print("РЕЗУЛЬТАТ REDUCE (сумма значений по каждому ключу):")
print("Формат: (ключ, сумма)")
print(reduce_output)

ИСХОДНЫЕ ДАННЫЕ: [(2, 3, 3), (0, 1, 3), (1, 2, 3), (3, 3, 2), (1, 3, 0), (2, 1, 3), (2, 0, 3), (1, 0, 2), (1, 1, 1), (0, 2, 2), (2, 1, 0), (1, 1, 2), (3, 2, 1), (2, 2, 1), (1, 2, 3), (1, 1, 0), (1, 2, 1), (3, 2, 1), (2, 1, 2), (2, 0, 2), (1, 2, 0), (2, 2, 1), (1, 2, 0), (3, 1, 2), (1, 1, 1), (0, 2, 1), (3, 1, 2), (0, 0, 2), (2, 0, 3), (2, 3, 3), (3, 1, 1), (0, 3, 1), (3, 0, 3), (2, 3, 1), (3, 3, 1), (0, 2, 3), (3, 2, 2), (1, 2, 1), (2, 0, 3), (2, 0, 2), (1, 3, 3), (3, 3, 2), (1, 3, 2), (0, 0, 3), (3, 2, 0), (2, 1, 0), (1, 2, 2), (0, 3, 3), (0, 0, 1), (1, 0, 2), (3, 3, 2), (2, 0, 2), (0, 0, 2), (1, 2, 3), (0, 2, 0), (3, 2, 3), (0, 3, 3), (2, 0, 1), (3, 1, 3), (2, 3, 1), (1, 0, 1), (0, 1, 1), (1, 1, 1), (0, 3, 2), (1, 1, 3), (1, 3, 1), (2, 3, 0), (0, 1, 0), (1, 3, 3), (1, 3, 0), (0, 0, 1), (2, 2, 3), (0, 3, 0), (1, 0, 0), (2, 2, 1), (0, 1, 2), (1, 3, 3), (0, 0, 3), (2, 3, 0), (0, 3, 2), (3, 1, 0), (2, 3, 3), (1, 0, 2), (3, 2, 2), (2, 3, 3), (2, 2, 0), (3, 2, 2), (2, 3, 2), (0, 3, 2), (1,

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [54]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [55]:
import numpy as np

# Размерности матриц
I = 2  # число строк в матрице A (small_mat)
J = 3  # число столбцов в A и число строк в B (общий размер для умножения)
K = 4 * 10  # число столбцов в матрице B (big_mat)

# Генерация случайных матриц
small_mat = np.random.rand(I, J)  # матрица A размером I×J
big_mat = np.random.rand(J, K)    # матрица B размером J×K

def RECORDREADER():
    """Читает элементы матрицы B и возвращает их как пары (ключ, значение).
    Ключ: (j, k) - координаты элемента в матрице B
    Значение: big_mat[j, k] - значение элемента"""
    for j in range(big_mat.shape[0]):      # строки матрицы B
        for k in range(big_mat.shape[1]):  # столбцы матрицы B
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    """Для каждого элемента матрицы B (j, k) со значением w,
    умножаем его на соответствующие элементы из столбца j матрицы A.
    Результат: пары ((i, k), w * A[i][j]) для всех i"""
    (j, k) = k1      # координаты элемента из B
    w = v1           # значение элемента из B
    for i in range(small_mat.shape[0]):  # для всех строк матрицы A
        # Умножаем элемент B на элемент A[i][j] и отправляем в пару с ключом (i, k)
        yield ((i, k), w * small_mat[i][j])

def REDUCE(key, values):
    """Суммирует все произведения для каждой ячейки результирующей матрицы.
    Ключ: (i, k) - координаты в результирующей матрице
    Значения: список всех w * A[i][j] для разных j
    Результат: ((i, k), сумма) - элемент результирующей матрицы"""
    (i, k) = key
    el_value = 0
    for v in values:
        el_value += v  # суммируем все произведения для данной позиции (i, k)
    yield ((i, k), el_value)

Проверьте своё решение

In [56]:
reference_solution = np.matmul(small_mat, big_mat)

# Запускаем наш MapReduce алгоритм
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
    """Преобразует результат MapReduce обратно в матрицу"""
    reduce_output = list(reduce_output)
    # Определяем размеры результирующей матрицы
    I = max(i for ((i, k), vw) in reduce_output) + 1
    K = max(k for ((i, k), vw) in reduce_output) + 1
    # Создаем пустую матрицу
    mat = np.empty(shape=(I, K))
    # Заполняем матрицу значениями из результатов Reduce
    for ((i, k), vw) in reduce_output:
        mat[i, k] = vw
    return mat

# Сравниваем наше решение с эталонным
result = np.allclose(reference_solution, asmatrix(solution))
print(f"Результат проверки: {result} (должно быть True)")

Результат проверки: True (должно быть True)


In [58]:
# Получаем результат MapReduce в виде списка
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))

# Находим максимальный индекс i (строки) в результирующей матрице
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [60]:
import numpy as np

# Задание размерностей матриц
I = 2        # число строк в первой матрице
J = 3        # число столбцов в первой матрице / число строк во второй
K = 4 * 10   # число столбцов во второй матрице

# Генерация случайных матриц
small_mat = np.random.rand(I, J)  # матрица A (I×J)
big_mat = np.random.rand(J, K)    # матрица B (J×K)

# Эталонное решение (прямое умножение матриц)
reference_solution = np.matmul(small_mat, big_mat)

def RECORDREADER():
    """Генератор, выдающий элементы обеих матриц с пометкой о принадлежности."""
    # Элементы первой матрицы (small_mat) с пометкой 0
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            yield ((0, i, j), small_mat[i, j])

    # Элементы второй матрицы (big_mat) с пометкой 1
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((1, j, k), big_mat[j, k])

def MAP_JOIN(k1, v1):
    """Преобразует элементы для соединения по общему индексу."""
    mat_num, idx1, idx2 = k1  # mat_num: 0 или 1, idx1: i или j, idx2: j или k
    value = v1

    if mat_num == 0:  # элемент из первой матрицы A[i,j]
        # Ключ = j (общий индекс), значение = (0, i, A[i,j])
        yield (idx2, (mat_num, idx1, value))
    else:  # элемент из второй матрицы B[j,k]
        # Ключ = j (общий индекс), значение = (1, k, B[j,k])
        yield (idx1, (mat_num, idx2, value))

def REDUCE_JOIN(key, values):
    """Соединяет элементы с одинаковым индексом j и вычисляет произведения."""
    # Разделяем значения на принадлежащие первой и второй матрицам
    from_first = [v for v in values if v[0] == 0]  # элементы из A
    from_second = [v for v in values if v[0] == 1]  # элементы из B

    # Для каждой пары (A[i,j], B[j,k]) вычисляем произведение
    for a_item in from_first:
        for b_item in from_second:
            # a_item: (0, i, A[i,j]), b_item: (1, k, B[j,k])
            # Результат: ((i, k), A[i,j] * B[j,k])
            yield ((a_item[1], b_item[1]), a_item[2] * b_item[2])

def MAP_MUL(k1, v1):
    """Пропускает пары (ключ, значение) без изменений."""
    yield (k1, v1)

def REDUCE_MUL(key, values):
    """Суммирует все значения для каждого ключа (i,k)."""
    result = 0
    for v in values:
        result += v
    yield (key, result)

def GET_JOINED():
    """Генератор, выдающий результаты первого MapReduce."""
    for item in joined:
        yield item

# Первый этап: соединение матриц и вычисление произведений
joined = MapReduce(RECORDREADER, MAP_JOIN, REDUCE_JOIN)

# Второй этап: суммирование произведений для получения итоговой матрицы
solution = MapReduce(GET_JOINED, MAP_MUL, REDUCE_MUL)

# Проверка совпадения с эталонным решением
is_correct = np.allclose(reference_solution, asmatrix(solution))
print(f"Результат проверки: {is_correct}")

Результат проверки: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [62]:
import numpy as np

# Определение размеров матриц
I = 2
J = 3
K = 4 * 10

# Генерация случайных матриц
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

# Эталонное решение через прямое умножение
reference_solution = np.matmul(small_mat, big_mat)

# Вспомогательные функции MapReduce
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    groups = {}
    for (key, value) in iterable:
        groups[key] = groups.get(key, []) + [value]
    return groups.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
    global reducers
    partitions = [dict() for _ in range(reducers)]
    for map_partition in map_partitions:
        for (key, value) in map_partition:
            p = partitions[PARTITIONER(key)]
            p[key] = p.get(key, []) + [value]
    return [(pid, sorted(p.items(), key=lambda x: x[0])) for (pid, p) in enumerate(partitions)]

def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
    # MAP фаза
    map_partitions = map(lambda reader: flatten(map(lambda rec: MAP(*rec), reader)), INPUTFORMAT())

    # COMBINE фаза (опционально)
    if COMBINER is not None:
        map_partitions = map(lambda mp: flatten(map(lambda g: COMBINER(*g), groupbykey(mp))), map_partitions)

    # SHUFFLE фаза
    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)

    # Подсет трафика
    traffic = sum(len(vs) for _, vs in flatten([p for _, p in reduce_partitions]))
    print(f"Передано по сети: {traffic} пар ключ-значение")

    # REDUCE фаза
    return map(lambda rp: (rp[0], flatten(map(lambda g: REDUCE(*g), rp[1]))), reduce_partitions)

def asmatrix(reduce_output):
    """Преобразует результат MapReduce обратно в матрицу"""
    output_list = list(reduce_output)
    I = max(i for ((i, k), _) in output_list) + 1
    K = max(k for ((i, k), _) in output_list) + 1
    mat = np.empty(shape=(I, K))
    for ((i, k), value) in output_list:
        mat[i, k] = value
    return mat

# INPUTFORMAT: подготовка данных
def INPUTFORMAT():
    # Элементы первой матрицы с пометкой 0
    first = []
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            first.append(((0, i, j), small_mat[i, j]))
    yield first

    # Элементы второй матрицы с пометкой 1
    second = []
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            second.append(((1, j, k), big_mat[j, k]))
    yield second

# MAP для соединения матриц
def MAP_JOIN(key, value):
    mat_num, idx1, idx2 = key
    if mat_num == 0:  # элемент из первой матрицы
        yield (idx2, (mat_num, idx1, value))  # ключ = j
    else:  # элемент из второй матрицы
        yield (idx1, (mat_num, idx2, value))  # ключ = j

# REDUCE для соединения матриц
def REDUCE_JOIN(key, values):
    from_first = [v for v in values if v[0] == 0]
    from_second = [v for v in values if v[0] == 1]
    for a in from_first:
        for b in from_second:
            yield ((a[1], b[1]), a[2] * b[2])  # ((i,k), A[i,j]*B[j,k])

# Получение результатов первого этапа
def GET_JOINED():
    for item in joined:
        yield item[1]  # возвращаем только данные, без номера партиции

# MAP для финального умножения
def MAP_MUL(key, value):
    yield (key, value)

# REDUCE для финального умножения (суммирование)
def REDUCE_MUL(key, values):
    total = 0
    for v in values:
        total += v
    yield (key, total)

# Параметры распределенной обработки
maps = 4
reducers = 2

print("ЭТАП 1: Соединение матриц и вычисление произведений")
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN, COMBINER=None)
joined = [(pid, list(data)) for pid, data in partitioned_output]
print(f"Получено {len(joined)} партиций после первого этапа")
print()

print("ЭТАП 2: Суммирование произведений для получения итоговой матрицы")
mul_output = MapReduceDistributed(GET_JOINED, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(pid, list(data)) for pid, data in mul_output]
print(f"Получено {len(pre_result)} партиций после второго этапа")
print()

# Собираем итоговый результат
solution = []
for pid, data in pre_result:
    for item in data:
        solution.append(item)

print("ИТОГОВАЯ ПРОВЕРКА")
print(f"Размерность полученной матрицы: {max(i for ((i,_),_) in solution)+1} x {max(k for ((_,k),_) in solution)+1}")
print(f"Совпадение с эталонным решением: {np.allclose(reference_solution, asmatrix(solution))}")

ЭТАП 1: Соединение матриц и вычисление произведений
Передано по сети: 126 пар ключ-значение
Получено 2 партиций после первого этапа

ЭТАП 2: Суммирование произведений для получения итоговой матрицы
Передано по сети: 240 пар ключ-значение
Получено 2 партиций после второго этапа

ИТОГОВАЯ ПРОВЕРКА
Размерность полученной матрицы: 2 x 40
Совпадение с эталонным решением: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [64]:
import numpy as np

# Определение размеров матриц
I = 2
J = 3
K = 4*10

# Генерация случайных матриц
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

# Получение эталонного решения через умножение матриц
reference_solution = np.matmul(small_mat, big_mat)

# Функция для "разглаживания" вложенных итерируемых объектов
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

# Функция для группировки элементов по ключу
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

# Функция для распределенной группировки элементов по ключу
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

# Функция для определения разделителя (по ключу)
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

# Функция для выполнения MapReduce на распределенных данных
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())

  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)

  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)  # shuffle

  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

# Функция для преобразования результата REDUCE в матрицу
def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output) + 1
  K = max(k for ((i,k), vw) in reduce_output) + 1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

# Генератор для вводных данных
def INPUTFORMAT():
  first_mat = []

  for i in range(small_mat.shape[0]):
    for j in range(small_mat.shape[1]):
      first_mat.append(((0, i, j), small_mat[i,j]))  # первая матрица

  global maps
  split_size = int(np.ceil(len(first_mat)/maps))

  for i in range(0, len(first_mat), split_size):
    yield first_mat[i:i+split_size]

  second_mat = []

  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      second_mat.append(((1, j, k), big_mat[j,k]))  # вторая матрица

  split_size = int(np.ceil(len(second_mat)/maps))

  for i in range(0, len(second_mat), split_size):
    yield second_mat[i:i+split_size]

# MAP функция для соединения матриц
def MAP_JOIN(k1, v1):
  (mat_num, i, j) = k1
  w = v1

  if mat_num == 0:
    yield (j, (mat_num, i, w))

  else:
    yield (i, (mat_num, j, w))

# REDUCE функция для соединения матриц
def REDUCE_JOIN(key, values):
  from_first_mat = [v for v in values if v[0] == 0]
  from_second_mat = [v for v in values if v[0] == 1]

  for f in from_first_mat:
    for s in from_second_mat:
      yield ((f[1], s[1]), f[2] * s[2])

# Генератор для получения соединенных данных
def GET_JOINED():

  for j in joined:
    print("aa", j)
    yield j[1]

# MAP функция для умножения значений
def MAP_MUL(k1, v1):
  yield (k1, v1)

# REDUCE функция для умножения значений
def REDUCE_MUL(key, values):
  res_val = 0

  for v in values:
    res_val += v
  yield (key, res_val)

maps = 3
reducers = 2

print("ЭТАП 1: СОЕДИНЕНИЕ МАТРИЦ")
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
print("РЕЗУЛЬТАТ СОЕДИНЕНИЯ:")
print(joined)
print()

print("ЭТАП 2: СУММИРОВАНИЕ ПРОИЗВЕДЕНИЙ")
mul_output = MapReduceDistributed(GET_JOINED, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]
print("РЕЗУЛЬТАТ СУММИРОВАНИЯ:")
print(pre_result)
print()

solution = []

for p in pre_result:
  for v in p[1]:
    solution.append(v)

print("ИТОГОВАЯ МАТРИЦА (элементы):")
print(solution)
print()

print("ПРОВЕРКА С ЭТАЛОННЫМ РЕШЕНИЕМ:")
print(np.allclose(reference_solution, asmatrix(solution)))

ЭТАП 1: СОЕДИНЕНИЕ МАТРИЦ
126 key-value pairs were sent over a network.
РЕЗУЛЬТАТ СОЕДИНЕНИЯ:
[(0, [((0, 0), np.float64(0.5133971679110008)), ((0, 1), np.float64(0.26471192807793625)), ((0, 2), np.float64(0.4146687808924577)), ((0, 3), np.float64(0.2557118405184104)), ((0, 4), np.float64(0.4743265546711734)), ((0, 5), np.float64(0.28806346526449744)), ((0, 6), np.float64(0.07162813958632305)), ((0, 7), np.float64(0.5980904676151604)), ((0, 8), np.float64(0.22543029048051963)), ((0, 9), np.float64(0.603286222045587)), ((0, 10), np.float64(0.09048688160308882)), ((0, 11), np.float64(0.3148533628967432)), ((0, 12), np.float64(0.43028796613880865)), ((0, 13), np.float64(0.09270711140581783)), ((0, 14), np.float64(0.3079937869790818)), ((0, 15), np.float64(0.14347161166458267)), ((0, 16), np.float64(0.31628684038671007)), ((0, 17), np.float64(0.21090007109847828)), ((0, 18), np.float64(0.5708591069726857)), ((0, 19), np.float64(0.31681470532280637)), ((0, 20), np.float64(0.04565939977755936